# Interconnected PMSM current control

`id_ref, iq_ref → current controller → vd,vq → SVPWM → duties → ideal inverter → Vabc → PMSM`

Physical motor feedback `Ia,Ib,Ic,theta → current feedback transform → id,iq` closes the current loop. Mechanical speed supplies back-EMF and decoupling feedforward.

Each block is a `control.NonlinearIOSystem`, assembled by `build_current_controlled_pmsm()` using explicit named connections. The controller has two continuous PI integral states; the PMSM has four internal states. The motor exposes only physical signals. The assembled drive also exposes controller and feedback diagnostics for plotting.

This is a continuous-time controller and averaged inverter simulation, without a PWM carrier, sample delay, or switching ripple. The output time grid specifies logging times, not controller update times.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import control as ct

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "motorsim/models/systems.py").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from motorsim.models.systems import build_current_controlled_pmsm


## Parameters and tuning

`Kp = L * bandwidth`, `Ki = R * bandwidth`, with bandwidth in rad/s. The controller includes dq decoupling and back-EMF feedforward matching the motor equations. Its output has a circular `V_bus / sqrt(3)` limit and back-calculation anti-windup (`Kaw`).

The equal-inductance example uses `Ke = pole_pairs * flux_linkage` and `Kt = Ke`, consistent with the model's `torque = 1.5 * Kt * iq`. There is no speed controller or field weakening; current tracking requires adequate bus voltage.


In [ ]:
pole_pairs = 4
flux_linkage = 0.025  # Wb
motor_params = {
    "Rm": 0.275, "Lq": 0.003, "np": pole_pairs,
    "Ke": pole_pairs * flux_linkage, "Kt": pole_pairs * flux_linkage,
    "J": 0.01, "b": 0.001,
}
inverter_params = {"V_bus": 48.0}
bandwidth = 2 * np.pi * 200
controller_gains = {
    "Kp": motor_params["Lq"] * bandwidth,
    "Ki": motor_params["Rm"] * bandwidth,
    "Kaw": bandwidth,
}
voltage_limit = inverter_params["V_bus"] / np.sqrt(3)
log_interval = 100e-6
duration = 0.8


In [ ]:
drive = build_current_controlled_pmsm(
    motor_params, inverter_params, controller_gains
)
print("External inputs:", drive.input_labels)
print("Outputs:", drive.output_labels)
print("States:", drive.state_labels)
# Controller integral_d, integral_q followed by motor iq, id, omega, theta.
x0 = np.zeros(drive.nstates)


## Simulate the connected drive

The positive iq command accelerates the rotor; the negative command brakes and reverses it. We simulate each constant-reference segment separately so the solver hits the command changes exactly and does not interpolate a step into a ramp. All motor integration and signal routing are handled by python-control.


In [ ]:
segments = [(0.0, 0.05, 0.0), (0.05, 0.4, 3.0), (0.4, duration, -3.0)]
time_parts, output_parts, reference_parts = [], [], []
for index, (start, stop, iq_ref) in enumerate(segments):
    times = np.linspace(start, stop, round((stop - start) / log_interval) + 1)
    inputs = np.vstack([np.zeros_like(times),
                        np.full_like(times, iq_ref),
                        np.zeros_like(times)])  # id_ref, iq_ref, load torque
    response = ct.input_output_response(
        drive, times, inputs, X0=x0, return_x=True,
        solve_ivp_kwargs={"rtol": 1e-7, "atol": 1e-9, "max_step": 1e-3},
    )
    x0 = response.states[:, -1]
    # Keep the sample at each step with the NEW controller command.
    keep = slice(None) if index == len(segments) - 1 else slice(None, -1)
    time_parts.append(response.time[keep])
    output_parts.append(response.outputs[:, keep].T)
    reference_parts.append(inputs[:2, keep].T)

time = np.concatenate(time_parts)
y = np.concatenate(output_parts)
log = {
    "physical": y[:, :5], "measured": y[:, 5:7],
    "voltage_dq": y[:, 7:9], "duties": y[:, 9:12],
    "v_abc": y[:, 12:15], "reference": np.concatenate(reference_parts),
}
print(f"Simulated {time[-1]:.2f} s with {drive.nstates} connected states")


In [ ]:
# Check tracking after each transient, using transformed physical feedback.
settled = ((time > 0.15) & (time < 0.38)) | (time > 0.55)
error = log["measured"] - log["reference"]
assert all(np.isfinite(values).all() for values in log.values())
assert np.max(np.abs(error[settled])) < 0.1, "Current tracking error exceeds 0.1 A"
assert np.max(np.abs(log["physical"][:, :3].sum(axis=1))) < 1e-10
assert log["duties"].min() >= -1e-12 and log["duties"].max() <= 1 + 1e-12
assert np.max(np.linalg.norm(log["voltage_dq"], axis=1)) <= voltage_limit + 1e-10
print("Settled maximum |id error|, |iq error| [A]:", np.max(np.abs(error[settled]), axis=0))
print("Final speed [rad/s]:", log["physical"][-1, 3])


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=5, cols=1, shared_xaxes=True,
                    subplot_titles=("Current tracking", "Controller voltages", "Mechanical speed",
                                    "Phase currents", "SVPWM duty cycles"))
for j, axis in enumerate(["d", "q"]):
    fig.add_trace(go.Scatter(x=time, y=log["reference"][:, j],
                            name=f"i{axis} reference", line=dict(dash="dash")), row=1, col=1)
    fig.add_trace(go.Scatter(x=time, y=log["measured"][:, j], name=f"i{axis} feedback"), row=1, col=1)
    fig.add_trace(go.Scatter(x=time, y=log["voltage_dq"][:, j], name=f"v{axis}"), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=log["physical"][:, 3], name="omega"), row=3, col=1)
for j, phase in enumerate("abc"):
    fig.add_trace(go.Scatter(x=time, y=log["physical"][:, j], name=f"I{phase}"), row=4, col=1)
    fig.add_trace(go.Scatter(x=time, y=log["duties"][:, j], name=f"d{phase}"), row=5, col=1)
for row, units in enumerate(["A", "V", "rad/s", "A", "duty"], start=1):
    fig.update_yaxes(title_text=units, row=row, col=1)
fig.update_xaxes(title_text="Time [s]", row=5, col=1)
fig.update_layout(height=1200, title="PMSM current loop: positive and negative iq commands")
fig.show()
